# Temporal Sales Dynamics in Retail Systems: A Data-Driven Analysis of Transaction Patterns and Demand Peaks

Project from Advanced Data Analytics (ITEC 4220)- Semester Project

#Abstract
This project investigates temporal purchasing behavior within a retail bakery environment using transactional sales data. The objective is to identify patterns in customer demand across time and evaluate how these patterns can inform operational decision-making.

The dataset includes variables such as transaction timestamps, product quantities, and pricing information. Data preprocessing was conducted to standardize time formats and extract meaningful temporal features, including hourly segmentation.

Exploratory analysis and statistical visualization techniques were applied to examine sales distributions, peak demand periods, and relationships between pricing and purchasing behavior. Additionally, clustering methods were implemented to segment transaction patterns and uncover latent structures within the data.

This analysis provides insight into how temporal dynamics influence retail performance and demonstrates how data-driven approaches can support optimization of inventory management, staffing, and production scheduling.

# Dataset Description and Preprocessing
This dataset contains transactional records from a retail bakery, including time of purchase, product quantity, and unit price.

The initial step involves loading the dataset and preparing key variables for analysis, particularly focusing on temporal features derived from transaction timestamps.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('https://raw.githubusercontent.com/anaya33/Bakery-Sales-Analysis/refs/heads/main/Bakery%20sales.csv')
df

,Unnamed: 0,date,time,ticket_number,article,Quantity,unit_price
0,0,2021-01-02,08:38,150040.0,BAGUETTE,1.0,"0,90 €"
1,1,2021-01-02,08:38,150040.0,PAIN AU CHOCOLAT,3.0,"1,20 €"
2,4,2021-01-02,09:14,150041.0,PAIN AU CHOCOLAT,2.0,"1,20 €"
3,5,2021-01-02,09:14,150041.0,PAIN,1.0,"1,15 €"
4,8,2021-01-02,09:25,150042.0,TRADITIONAL BAGUETTE,5.0,"1,20 €"
...,...,...,...,...,...,...,...
234000,511387,2022-09-30,18:52,288911.0,COUPE,1.0,"0,15 €"
234001,511388,2022-09-30,18:52,288911.0,BOULE 200G,1.0,"1,20 €"
234002,511389,2022-09-30,18:52,288911.0,COUPE,2.0,"0,15 €"
234003,511392,2022-09-30,18:55,288912.0,TRADITIONAL BAGUETTE,1.0,"1,30 €"


# Data Cleaning


In [ ]:
columns_to_keep = [
    "time",
    "Quantity",
    "unit_price"
]

# print(df.columns)

# df = df.dropna(subset=["time"])
# df = df.dropna(subset=["Quantity", "unit_price"])

# df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
# df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

# df["time"] = pd.to_datetime(df["time"], format="%H:%M", errors="coerce")
# df["hour"] = df["time"].dt.hour

df.describe()

In [ ]:
df.head()

In [ ]:
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")
df["time"] = pd.to_datetime(df["time"], format="%H:%M", errors="coerce")

In [ ]:
df["hour"] = df["time"].dt.hour

df = df.dropna(subset=["hour", "Quantity", "unit_price"])

df.head()

# Investigating the Relationship, if any, Between Time, Quantity, and Unit Price

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df["hour"], bins=24, kde=False)
plt.title("Sales Distribution by Hour of the Day (24-hour format)")
plt.xlabel("Hour of the Day")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x=df["unit_price"], y=df["Quantity"], alpha=0.6)
plt.title("Relationship Between Unit Price and Sales Volume")
plt.xlabel("Unit Price ($)")
plt.ylabel("Quantity Sold")
plt.grid(axis="both", linestyle="--", alpha=0.5)
plt.show()

# K-Means Testing with and without SKLearn

In [ ]:
df_kmeans = df[["Quantity", "unit_price"]].dropna()

data = df_kmeans.values

sample_data = data[:10000]

In [ ]:
k = 3
max_iters = 100
tolerance = 1e-4

np.random.seed(42)
initial_centroids = sample_data[np.random.choice(sample_data.shape[0], k, replace=False)]

In [ ]:
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

In [ ]:
centroids = initial_centroids

for iteration in range(max_iters):
    clusters = [[] for _ in range(k)]

    for point in sample_data:
        distances = [euclidean_distance(point, centroid) for centroid in centroids]
        closest_centroid = np.argmin(distances)
        clusters[closest_centroid].append(point)

    new_centroids = np.array([
        np.mean(cluster, axis=0) if cluster else centroid
        for cluster, centroid in zip(clusters, centroids)
    ])

    if np.all(np.abs(new_centroids - centroids) < tolerance):
        break

    centroids = new_centroids

In [ ]:
sample_labels = np.zeros(sample_data.shape[0], dtype=int)

for cluster_index, cluster in enumerate(clusters):
    for point in cluster:
        index = np.where((sample_data == point).all(axis=1))[0][0]
        sample_labels[index] = cluster_index

In [ ]:
sample_df = pd.DataFrame(sample_data, columns=["Quantity", "unit_price"])
sample_df["Cluster"] = sample_labels

sample_df.head()

In [ ]:
plt.figure(figsize=(10,6))

for cluster in np.unique(sample_labels):
    cluster_points = sample_df[sample_df["Cluster"] == cluster]
    plt.scatter(cluster_points["unit_price"], cluster_points["Quantity"], label=f"Cluster {cluster}", alpha=0.6)

plt.xlabel("Unit Price")
plt.ylabel("Quantity Sold")
plt.title("Manual K-Means Clustering")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(sample_data)

sample_df["Cluster_SKLearn"] = kmeans.labels_

sample_df.head()

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(data=sample_df, x="unit_price", y="Quantity", hue="Cluster_SKLearn", palette="deep", alpha=0.7)
plt.title("K-Means Clustering using SKLearn")
plt.xlabel("Unit Price")
plt.ylabel("Quantity Sold")
plt.show()

# Correlation Matrix

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df[["hour","Quantity","unit_price"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

## K-NN Test

df_knn = df[["Quantity","unit_price"]].dropna().copy()

df_knn["Label"] = (df_knn["Quantity"] > df_knn["Quantity"].median()).astype(int)

df_train = df_knn.iloc[:20]
df_test = df_knn.iloc[[3]]

In [ ]:
test_quantity = df_test.iloc[0]["Quantity"]
test_unit_price = df_test.iloc[0]["unit_price"]

distances = np.sqrt(
    ((df_train["Quantity"] - test_quantity) ** 2) +
    ((df_train["unit_price"] - test_unit_price) ** 2)
)

k = 5
nearest_neighbors = df_train.iloc[distances.nsmallest(k).index]

nearest_neighbors

In [ ]:
most_common_label = nearest_neighbors["Label"].mode()[0]

print("Predicted class:", most_common_label)

# K-NN using SKLearn

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X_train = df_train[["Quantity","unit_price"]]
y_train = df_train["Label"]

X_test = df_test[["Quantity","unit_price"]]

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

prediction = knn.predict(X_test)

print("Predicted class:", prediction[0])

# Principal Component Analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pca_df = df[["Quantity","unit_price","hour"]].dropna()

scaler = StandardScaler()
scaled_data = scaler.fit_transform(pca_df)

In [ ]:
pca = PCA()
pca.fit(scaled_data)

explained_variance_ratio = pca.explained_variance_ratio_
cumulative_variance = explained_variance_ratio.cumsum()

explained_variance_ratio, cumulative_variance

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(range(1, len(explained_variance_ratio)+1), explained_variance_ratio)
plt.step(range(1, len(cumulative_variance)+1), cumulative_variance, where="mid")
plt.xlabel("Principal Components")
plt.ylabel("Explained Variance Ratio")
plt.title("Explained Variance by Principal Components")
plt.show()

In [ ]:
pca = PCA(n_components=2)
transformed_data = pca.fit_transform(scaled_data)

transformed_df = pd.DataFrame(transformed_data, columns=["PC1","PC2"])

transformed_df.head()

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(transformed_df["PC1"], transformed_df["PC2"], alpha=0.7)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Projection")
plt.show()

# Interpretation
The results from this analysis suggest that bakery transaction behavior varies across time and may be influenced by both pricing and